In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

dataset = pd.read_csv("diabetes_data_upload.csv")

feature_cols = ["Polyuria", "Gender", "sudden weight loss", "partial paresis"]

col_to_pd = dataset[feature_cols]
for col in col_to_pd:
    dataset[col] = pd.factorize(dataset[col])[0]

map_target = {'Positive': 1, "Negative" : 0}
dataset['class'] = dataset['class'].map(map_target)

dataset = dataset.drop_duplicates()

def stratified_split(dataset, target_column, training_size=0.8, random_state=42):
    np.random.seed(random_state)
    train_list, test_list = [], []

    for class_value in dataset[target_column].unique():
        class_data = dataset[dataset[target_column] == class_value]
        class_data = class_data.sample(frac=1, random_state=random_state)

        split_idx = int(len(class_data) * training_size)
        train_list.append(class_data.iloc[:split_idx])
        test_list.append(class_data.iloc[split_idx:])

    train_set = pd.concat(train_list).reset_index(drop=True)
    test_set = pd.concat(test_list).reset_index(drop=True)
    return train_set, test_set

dataset_training, dataset_testing = stratified_split(dataset, "class")

X_train = dataset_training[feature_cols].to_numpy()
X_test = dataset_testing[feature_cols].to_numpy()

y_train = dataset_training["class"].to_numpy()
y_test = dataset_testing["class"].to_numpy()

class TreeEnsemble:
    def __init__(self, max_depth=5, feature_names=feature_cols, mode="single", n_trees=10, sample_ratio=0.8, feature_ratio=0.8):
        self.max_depth = max_depth
        self.mode = mode
        self.feature_names = feature_names
        self.n_trees = n_trees
        self.sample_ratio = sample_ratio
        self.feature_ratio = feature_ratio
        self.trees = [] if self.mode == "ensemble" else None

    def most_common_value(self, arr):
        unique, counts = np.unique(arr, return_counts=True)
        return unique[np.argmax(counts)]
    
    def entropy(self, y):
        unique, counts = np.unique(y, return_counts=True)
        probs = counts / len(y)
        return -np.sum(probs * np.log2(probs * 1e-9))
    
    def information_gain(self, parent, left, right):
        return self.entropy(parent) - (len(left) / len(parent) * self.entropy(left) + len(right) / len(parent) * self.entropy(right))
    
    def _build_tree(self, X, y, depth=0):
        if depth >= self.max_depth or len(np.unique(y)) == 0:
            return self.most_common_value(y)
        
        best_feature, best_split, best_gain = None, None, -1
        for feature in range(X.shape[1]):
            unique_values = np.unique(X[:, feature])
            for value in unique_values:
                left_mask = X[:, feature] <= value
                right_mask = X[:, feature] > value

                if len(y[left_mask]) == 0 or len(y[right_mask]) == 0:
                    continue

                gain = self.information_gain(y, y[left_mask], y[right_mask])
                if gain > best_gain:
                    best_feature, best_split, best_gain = feature, value, gain

        if best_gain == -1:
            return self.most_common_value(y)
        
        left_mask = X[:, best_feature] <= best_split
        right_mask = X[:, best_feature] > best_split

        return {
            "feature": best_feature,
            "split": best_split,
            "left": self._build_tree(X[left_mask], y[left_mask], depth + 1),
            "right": self._build_tree(X[right_mask], y[right_mask], depth + 1)
        }
    
    def fit(self, X, y):
        if self.mode == "single":
            self.trees = self._build_tree(X, y)

        else:
            self.trees = []
            n_samples = int(len(X) * self.sample_ratio)
            n_features = int(X.shape[1] * self.feature_ratio)

            for _ in range(self.n_trees):
                sample_idx = np.random.choice(len(X), n_samples, replace=True)
                feature_idx = np.random.choice(X.shape[1], n_features, replace=False)
                X_sample, y_sample = X[sample_idx][:, feature_idx], y[sample_idx]
                tree = self._build_tree(X_sample, y_sample)
                self.trees.append((tree, feature_idx))

    def _predict_sample(self, tree, sample):
        if not isinstance(tree, dict):
            return tree
        
        feature = tree["feature"]
        split = tree["split"]

        if sample[feature] <= split:
            return self._predict_sample(tree["left"], sample)
        else:
            return self._predict_sample(tree["right"], sample)
        
    def predict(self, X):
        if self.mode == "single":
            return np.array([self._predict_sample(self.trees, sample) for sample in X])
        
        predictions = []
        for tree, feature_idx in self.trees:
            preds = [self._predict_sample(tree, sample[feature_idx]) for sample in X]
            predictions.append(preds)

        predictions = np.array(predictions)
        final_preds = [np.bincount(predictions[:, i]).argmax() for i in range(predictions.shape[1])]
        return np.array(final_preds)
        
    def print_tree(self, tree = None, depth = 0):
        if tree is None:
            tree = self.trees if self.mode == 'single' else self.trees[0][0]
        
        if not isinstance(tree, dict):
            print("  " * depth + f"Predict: {tree}")
            return 
        
        feature_name = self.feature_names[tree['feature']] if self.feature_names else f"{tree['feature']}"
        print("  " * depth + f"({feature_name}) <= {tree['split']}")

        print("  " * depth + "-> Left:")
        self.print_tree(tree['left'], depth + 1)

        print("  " * depth + "-> Right:")
        self.print_tree(tree['right'], depth + 1)
        
    def plot_tree(self, tree = None, x = 0.5, y = 1, dx = 0.25, dy = 0.15 , ax = None):

        if ax is None:
            fig, ax = plt.subplots(figsize = (30, 40))

        ax.axis("off")
        
        if tree is None:
            tree = self.trees if self.mode == 'single' else self.trees[0][0]
        

        def _plot_node(tree, x, y, dx, dy):

            if not isinstance(tree, dict):
                ax.text(x, y, f'{tree}', bbox = dict(facecolor = "lightblue", alpha = 0.5), ha = 'center', va = "center")
                return tree

            feature_name = self.feature_names[tree['feature']] if self.feature_names else f"{tree['feature']}"
            ax.text(x, y, f"{feature_name} <= {tree['split']}", bbox = dict(facecolor = "lightblue", alpha = 0.5), ha = 'center', va = "center")

            x_left, x_right = x - dx, x + dx
            y_next = y - dy

            ax.plot([x, x_left], [y, y_next], 'k-')
            ax.plot([x, x_right], [y, y_next], 'k-')

            ax.text((x + x_left)/2, (y + y_next)/2, "True", color = 'green',fontsize = 13)
            ax.text((x + x_right)/2, (y + y_next)/2, "False", color = 'red',fontsize = 13)

            _plot_node(tree['left'], x_left, y_next, dx/2, dy)
            _plot_node(tree['right'], x_right, y_next, dx/2, dy)

        _plot_node(tree, x, y, dx, dy)

    def plot_forest(self, num_trees):
        num_trees = min(num_trees, len(self.trees))

        for i in range(num_trees):

            tree, _ = self.trees[i]
            fig, ax = plt.subplots(figsize = (30,40))
            self.plot_tree(tree, ax = ax)
            plt.title(f"Tree: {i + 1}")
            plt.show()

    def trace_decision_path(self, tree, sample, depth = 0):
        if not isinstance(tree, dict):
            print("  " * depth + f"==> Predict: {tree}")
            return tree
        
        split = tree['split']
        feature = tree['feature']
        decision = "Left" if sample[feature] <= split else "Right"

        print("  " * depth + f"{self.feature_names[feature] if self.feature_names else f'(Feature: {feature})'} <= {split} {decision}")
        next_branch = tree['left'] if sample[feature] <= split else tree['right']

        return self.trace_decision_path(next_branch, sample, depth + 1)

    def predict_with_trace(self, X):
        predictions = []

        for i, sample in enumerate(X):
            print(f"\nSample: {i + 1}")
            
            if self.mode == 'single':
                pred = self.trace_decision_path(self.trees, sample)
            
            else:
                tree_pred = []
                for j, (tree, feature_idx) in enumerate(self.trees):
                    print(f"\nTree: {j + 1}")
                    reduced_sample = np.array(sample)[feature_idx]
                    tree_pred.append(self.trace_decision_path(tree ,reduced_sample))

                pred = np.bincount(tree_pred).argmax()
                print(f"Majority Vote: {pred}")
            
            predictions.append(pred)

        return np.array(predictions)

model_tree = TreeEnsemble(mode="single", max_depth=7)
model_tree.fit(X_train, y_train)

y_pred_tree_train = model_tree.predict(X_train)
y_pred_tree_test = model_tree.predict(X_test)